In [ ]:
#!/usr/bin/env python3

import json
import os
import argparse
from collections import Counter, defaultdict
from pathlib import Path

# Load exclusion lists
exclude_ids_by_task = { # these samples lack of responses for reasoning
    "vague": ["3I02618YA14S7SHVOPAYBZTIDARUPP", "3KIBXJ1WD6SWJW0IFBTHGCFU1SKOKK", "3NPI0JQDAP3D7F26OKKO637GUJPPT8", "3SITXWYCNW7IK2AGAP3K0MNXQ3DBXR"],
    "reverse": ["39L1G8WVWRP5R6LAO337NULKX9931E"],
    "biased": [],
    "fake": ["9929"]
}

def load_results_from_directory(results_dir, task, resp_flag):
    """Load all JSON result files from a directory"""
    results = []
    json_files = list(Path(results_dir).glob("result_*.json"))
    
    print(f"Found {len(json_files)} result files")
    
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                results.append(data)
        except Exception as e:
            print(f"Error loading file {json_file}: {e}")

    # Exclude results based on task-specific criteria
    if resp_flag:
        if task in exclude_ids_by_task:
            exclude_ids = set(exclude_ids_by_task[task]) 
            results = [r for r in results if r.get('sample_id') not in exclude_ids]

    return results


def analyze_assessments(results):
    """Analyze distribution of assessments"""
    assessments = []
    for result in results:
        parsed_response = result.get('parsed_response', {})
        assessment = parsed_response.get('response', 'unknown')
        assessments.append(assessment)
    
    assessment_counts = Counter(assessments)
    return assessment_counts


def analyze_violated_categories(results):
    """Analyze distribution of violated_categories"""
    all_categories = []
    category_combinations = []
    
    for result in results:
        parsed_response = result.get('parsed_response', {})
        parsed_response = parsed_response.get('parsed_response', {})
        category = parsed_response.get('category', [])
        
        # Record single category
        # for category in violated_categories:
        #     all_categories.append(category)
        all_categories.append(category)
        
        # Record category combinations
        if category:
            combo = sorted(category)
            category_combinations.append(tuple(combo))
        else:
            category_combinations.append(('None',))
    
    single_category_counts = Counter(all_categories)
    combination_counts = Counter(category_combinations)
    
    return single_category_counts, combination_counts


def print_statistics(results):
    """Print computed statistics"""
    print(f"\n{'='*60}")
    print(f"Overall statistics - {len(results)} samples")
    print(f"{'='*60}")
    
    # 1. Assessment distribution
    assessment_counts = analyze_assessments(results)
    print("\nAssessment distribution:")
    print("-" * 30)
    total_samples = sum(assessment_counts.values())
    for assessment, count in assessment_counts.most_common():
        percentage = (count / total_samples) * 100
        print(f"  {assessment:10}: {count:4} ({percentage:5.1f}%)")



In [ ]:
for model in ['shield', 'shield_resp', 'shield_single', 'shield_resp_single']:
    for task in ['vague', 'reverse', 'biased', 'fake']:
        results_dir = f'./results_{task}_{model}'

    if model.endswith('resp'):
        resp_flag = True
    else:
        resp_flag = False

    if not os.path.exists(results_dir):
        print(f"Error: directory {results_dir} does not exist")

    # Load results
    print(f"Loading results from {results_dir}...")
    results = load_results_from_directory(results_dir, task=task, resp_flag=resp_flag)

    if not results:
        print("Error: No valid result files found")

    # Print statistics
    print_statistics(results)

Loading results from ./results_vague_shield_single_resp...
Found 2500 result files

Overall statistics - 2496 samples

Assessment distribution:
------------------------------
  No        : 2095 ( 83.9%)
  Yes       :  401 ( 16.1%)
Loading results from ./results_reverse_shield_single_resp...
Found 2500 result files

Overall statistics - 2499 samples

Assessment distribution:
------------------------------
  No        : 2494 ( 99.8%)
  Yes       :    5 (  0.2%)
Loading results from ./results_biased_shield_single_resp...
Found 2800 result files

Overall statistics - 2800 samples

Assessment distribution:
------------------------------
  No        : 2643 ( 94.4%)
  Yes       :  157 (  5.6%)
Loading results from ./results_fake_shield_single_resp...
Found 2800 result files

Overall statistics - 2799 samples

Assessment distribution:
------------------------------
  No        : 2786 ( 99.5%)
  Yes       :   13 (  0.5%)
